In [2]:
import duckdb
import pandas as pd

# Import jupysql Jupyter extension to create SQL cells
%load_ext sql

%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [3]:
%sql duckdb:///:memory:

In [4]:
%%sql
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS marketing;

CREATE TABLE sales AS SELECT * FROM read_csv_auto('tables/sales.csv');
CREATE TABLE customers AS SELECT * FROM read_csv_auto('tables/customers.csv');
CREATE TABLE marketing AS SELECT * FROM read_csv_auto('tables/marketing.csv');

,Success


In [5]:
%sql DESCRIBE

,Success


## Example
Tell me all the product names.

In [6]:
%%sql

select
    item
from
    sales
group by
    item

,item
0,Bed
1,Dinner table
2,Sofa


## Question 1
Tell me the top 10 countries by their number of sales.

In [7]:
%%sql
Select c.country from Sales s
JOIN customers c on c.Id=s.customer_ID Group by c.country Order by Count(item) desc limit 10;


,country
0,Germany
1,USA
2,Denmark
3,France
4,Canada
5,Austria
6,UK
7,Sweden
8,Spain
9,Norway


## Question 1.5
Tell me the top 9 countries by sales, plus a 10th row for “all others”.

In [8]:
%%sql
WITH CTE AS (
Select c.country from Sales s
JOIN customers c on c.Id=s.customer_ID Group by c.country Order by Count(item) desc limit 9)
Select * from CTE
UNION ALL
Select 'all others';

,country
0,Germany
1,USA
2,Denmark
3,France
4,Canada
5,Austria
6,UK
7,Sweden
8,Spain
9,all others


## Question 2
A repeat customer is someone who makes 2+ purchases, but 2+ on the same day doesn’t count. Tell me how many repeat customers we have.

In [9]:
%%sql
WITH RepeatCustomer  AS (  
Select date,customer_id from Sales group by all)
Select Count(customer_id) from RepeatCustomer Having Count(date)>2

,count(customer_id)
0,9947


## Question 3
Tell me the count of customers with only organic orders on a last touch attribution basis.

In [10]:
%%sql
WITH last_touch AS (
    SELECT
        customer_email,
        channel,
        ROW_NUMBER() OVER (
            PARTITION BY customer_email
            ORDER BY date DESC
        ) AS rn
    FROM marketing
)

SELECT COUNT(DISTINCT customer_email) AS organic_only_customers
FROM last_touch
WHERE rn = 1
  AND LOWER(channel) = 'organic';

,organic_only_customers
0,1861
